In [ ]:
# ============================================================================
# KAGGLE ENVIRONMENT SETUP
# ============================================================================
# This notebook is designed to run on Kaggle's platform, which provides:
# - Pre-installed data science libraries (numpy, pandas, sklearn, etc.)
# - GPU acceleration for deep learning
# - Access to datasets via /kaggle/input/ directory

import numpy as np  # For numerical operations and array manipulation
import pandas as pd  # For data manipulation and CSV file operations

# Display all available input files in the Kaggle environment
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# NOTE: Kaggle provides 20GB of persistent storage in /kaggle/working/
# Use /kaggle/temp/ for temporary files that won't be saved

# 📚 Deep Learning Template: MNIST Digit Recognition

## Overview
This notebook demonstrates a complete deep learning workflow using PyTorch:
1. **Data Loading & Preprocessing** - Preparing image data for neural networks
2. **Custom Dataset Creation** - Building flexible data pipelines
3. **CNN Architecture** - Designing convolutional neural networks
4. **Training Loop** - Implementing the learning process
5. **Prediction & Submission** - Making predictions on new data

## Import Libraries and Load Data
We'll use PyTorch as our deep learning framework

In [ ]:
# ============================================================================
# PYTORCH & DEEP LEARNING IMPORTS
# ============================================================================

import torch  # Core PyTorch library for tensor operations
import torch.nn as nn  # Neural network modules (layers, loss functions, etc.)
import torch.optim as optim  # Optimization algorithms (Adam, SGD, etc.)
from torch.utils.data import Dataset, DataLoader  # Data handling utilities

import pandas as pd  # For loading CSV files
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For visualization
from torchvision import transforms  # Image transformation utilities

# Check PyTorch version and CUDA availability
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# DATA LOADING & PREPROCESSING
# ============================================================================

# Load CSV Files
# train.csv contains labeled digit images (0-9) for training
# test.csv contains unlabeled images for prediction
train_data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_data = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Training data shape: {train_data.shape}")

# Separate features (pixel values) and labels (digits)
# Features: All columns except 'label' (784 pixel values per image)
# Labels: The 'label' column (digits 0-9)
X_train = train_data.drop('label', axis=1).values
y_train = train_data['label'].values
X_test = test_data.values

# Data Preprocessing:
# 1. Reshape: Convert flat 784-pixel vectors into 28x28 images
#    -1: Automatically determine batch size
#    1: Number of channels (grayscale = 1, RGB = 3)
#    28x28: Image dimensions
# 2. Convert to float32: PyTorch uses float32 for neural network computations
# 3. Normalize: Scale pixel values from [0, 255] to [0, 1]
#    Why? Neural networks train better with normalized inputs
X_train = X_train.reshape(-1, 1, 28, 28).astype('float32') / 255.0
X_test = X_test.reshape(-1, 1, 28, 28).astype('float32') / 255.0

print(f"Processed training data shape: {X_train.shape}")
print(f"Processed test data shape: {X_test.shape}")
print(f"Label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

# 🔧 Create Custom Dataset and DataLoader

## Why Custom Datasets?
- **Flexibility**: Handle any data format (CSV, images, text, etc.)
- **Memory Efficiency**: Load data on-demand rather than all at once
- **Integration**: Works seamlessly with PyTorch's training pipeline

## DataLoader Benefits:
- **Batching**: Process multiple samples simultaneously for efficiency
- **Shuffling**: Randomize data order to prevent overfitting
- **Parallel Loading**: Use multiple workers to speed up data loading

In [ ]:
# ============================================================================
# CUSTOM DATASET CLASS
# ============================================================================

class MNISTDataset(Dataset):
    """
    Custom Dataset for MNIST digit images.
    
    A PyTorch Dataset must implement three methods:
    1. __init__: Initialize the dataset (store data, labels, transforms)
    2. __len__: Return the total number of samples
    3. __getitem__: Return a single sample given an index
    """
    
    def __init__(self, data, labels=None, transform=None):
        """
        Args:
            data (numpy.ndarray): Image data, shape (N, C, H, W)
            labels (numpy.ndarray, optional): Labels for images
            transform (callable, optional): Optional transform to apply to images
        """
        # Convert numpy arrays to PyTorch tensors
        self.data = torch.FloatTensor(data)
        self.labels = torch.LongTensor(labels) if labels is not None else None
        self.transform = transform
    
    def __len__(self):
        """Return the total number of samples in the dataset."""
        return len(self.data)
    
    def __getitem__(self, idx):
        """
        Retrieve a single sample from the dataset.
        
        Args:
            idx (int): Index of the sample to retrieve
            
        Returns:
            tuple: (image, label) if labels exist, otherwise just image
        """
        x = self.data[idx]
        
        # Apply transformations if specified (e.g., augmentation)
        if self.transform:
            x = self.transform(x)
        
        # Return (image, label) pair for training, or just image for testing
        if self.labels is not None:
            return x, self.labels[idx]
        return x


# ============================================================================
# CREATE DATASETS AND DATALOADERS
# ============================================================================

# Create dataset objects
train_dataset = MNISTDataset(X_train, y_train)
test_dataset = MNISTDataset(X_test)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Create DataLoader objects
# DataLoader wraps a Dataset and provides:
# - Batching: Group samples into batches for efficient processing
# - Shuffling: Randomize sample order each epoch (training only)
# - Parallel loading: Use multiple CPU cores to load data

train_loader = DataLoader(
    train_dataset,
    batch_size=64,      # Process 64 images at a time
    shuffle=True,       # Randomize order each epoch (prevents overfitting)
    num_workers=0       # Number of parallel data loading processes (0 = main process)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,      # Process 64 images at a time
    shuffle=False,      # Keep original order for test data
    num_workers=0
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# 🧠 Build a Convolutional Neural Network (CNN) Architecture

## Why CNNs for Images?
- **Spatial Awareness**: Preserve spatial relationships between pixels
- **Feature Learning**: Automatically learn visual features (edges, shapes, patterns)
- **Parameter Efficiency**: Share weights across the image (fewer parameters than fully connected)

## Our Architecture:
1. **Convolutional Layers**: Extract visual features
2. **Pooling Layers**: Reduce spatial dimensions, increase computational efficiency
3. **Dropout Layers**: Prevent overfitting by randomly disabling neurons
4. **Fully Connected Layers**: Combine features for final classification

In [ ]:
# ============================================================================
# CNN ARCHITECTURE
# ============================================================================

class DigitCNN(nn.Module):
    """
    Convolutional Neural Network for digit classification.
    
    Architecture Overview:
    - Input: 1x28x28 (grayscale image)
    - Conv1 → Pool → Conv2 → Pool → Dropout
    - Flatten → FC1 → Dropout → FC2
    - Output: 10 classes (digits 0-9)
    """
    
    def __init__(self):
        """Initialize all layers of the network."""
        super(DigitCNN, self).__init__()
        
        # ====================================================================
        # CONVOLUTIONAL LAYERS (Feature Extraction)
        # ====================================================================
        
        # First convolutional layer
        # Input: 1 channel (grayscale), Output: 32 feature maps
        # kernel_size=3: Use 3x3 filters to detect small patterns
        # padding=1: Add 1 pixel border to maintain spatial dimensions
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, 
                               kernel_size=3, padding=1)
        # After conv1: 32x28x28 (32 feature maps, same spatial size due to padding)
        
        # Second convolutional layer
        # Input: 32 channels, Output: 64 feature maps
        # Deeper layers learn more complex features
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, 
                               kernel_size=3, padding=1)
        # After conv2: 64x14x14 (after first pooling)
        
        # Max Pooling layer
        # Reduces spatial dimensions by taking maximum value in 2x2 windows
        # Helps with: translation invariance, computational efficiency
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Effect: Reduces 28x28 → 14x14, then 14x14 → 7x7
        
        # First Dropout layer (regularization)
        # Randomly sets 25% of activations to zero during training
        # Prevents overfitting by forcing the network to learn robust features
        self.dropout1 = nn.Dropout(p=0.25)
        
        # ====================================================================
        # FULLY CONNECTED LAYERS (Classification)
        # ====================================================================
        
        # First fully connected layer
        # Input: 64 * 7 * 7 = 3136 features (flattened feature maps)
        # Output: 128 neurons (compressed representation)
        self.fc1 = nn.Linear(in_features=64 * 7 * 7, out_features=128)
        
        # Second Dropout layer
        # Higher dropout rate (50%) for fully connected layers
        # FC layers are more prone to overfitting
        self.dropout2 = nn.Dropout(p=0.5)
        
        # Output layer
        # Input: 128 features, Output: 10 classes (digits 0-9)
        # No activation here - we'll use CrossEntropyLoss which includes softmax
        self.fc2 = nn.Linear(in_features=128, out_features=10)
        
    def forward(self, x):
        """
        Forward pass: Define how data flows through the network.
        
        Args:
            x (torch.Tensor): Input batch of images, shape (batch_size, 1, 28, 28)
            
        Returns:
            torch.Tensor: Raw class scores (logits), shape (batch_size, 10)
        """
        # ====================================================================
        # FEATURE EXTRACTION
        # ====================================================================
        
        # First convolutional block: Conv → ReLU → Pool
        # Shape: (batch, 1, 28, 28) → (batch, 32, 28, 28) → (batch, 32, 14, 14)
        x = self.conv1(x)           # Apply first convolution
        x = torch.relu(x)           # Apply ReLU activation (introduces non-linearity)
        x = self.pool(x)            # Downsample by 2x (28x28 → 14x14)
        
        # Second convolutional block: Conv → ReLU → Pool → Dropout
        # Shape: (batch, 32, 14, 14) → (batch, 64, 14, 14) → (batch, 64, 7, 7)
        x = self.conv2(x)           # Apply second convolution
        x = torch.relu(x)           # Apply ReLU activation
        x = self.pool(x)            # Downsample by 2x (14x14 → 7x7)
        x = self.dropout1(x)        # Apply dropout for regularization
        
        # ====================================================================
        # CLASSIFICATION
        # ====================================================================
        
        # Flatten feature maps into a 1D vector
        # Shape: (batch, 64, 7, 7) → (batch, 3136)
        # This prepares data for fully connected layers
        x = x.view(-1, 64 * 7 * 7)
        
        # First fully connected layer: Linear → ReLU → Dropout
        # Shape: (batch, 3136) → (batch, 128)
        x = self.fc1(x)             # Apply first FC layer
        x = torch.relu(x)           # Apply ReLU activation
        x = self.dropout2(x)        # Apply dropout
        
        # Output layer: Linear (no activation)
        # Shape: (batch, 128) → (batch, 10)
        # Returns raw scores (logits) for each class
        x = self.fc2(x)
        
        return x


# ============================================================================
# MODEL SUMMARY
# ============================================================================

# Create a sample model to display architecture
sample_model = DigitCNN()
print(sample_model)
print(f"\nTotal parameters: {sum(p.numel() for p in sample_model.parameters()):,}")

# ⚙️ Set Up Training Configuration

## Key Components:
1. **Device**: Use GPU if available for faster training
2. **Model**: Initialize and move to device
3. **Loss Function**: How we measure prediction error
4. **Optimizer**: How we update weights to minimize loss
5. **Scheduler**: Dynamically adjust learning rate during training

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

# Device Selection: Use GPU if available, otherwise CPU
# GPU training is typically 10-100x faster for neural networks
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Initialize model and move to device
model = DigitCNN().to(device)
print(f"Model moved to {device}")

# ============================================================================
# LOSS FUNCTION
# ============================================================================

# CrossEntropyLoss: Standard loss for multi-class classification
# Combines LogSoftmax and NLLLoss in one operation
# - Converts raw scores (logits) to probabilities
# - Measures difference between predicted and true distributions
criterion = nn.CrossEntropyLoss()

print(f"Loss function: {criterion.__class__.__name__}")

# ============================================================================
# OPTIMIZER
# ============================================================================

# Adam (Adaptive Moment Estimation): Popular optimizer that:
# - Adapts learning rate for each parameter
# - Uses momentum for faster convergence
# - Works well with minimal hyperparameter tuning
# 
# lr=0.001: Learning rate (how big each weight update is)
# - Too high: Training becomes unstable
# - Too low: Training is very slow
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")

# ============================================================================
# LEARNING RATE SCHEDULER (Optional but Recommended)
# ============================================================================

# ReduceLROnPlateau: Reduce learning rate when training plateaus
# This helps fine-tune the model in later epochs
# 
# mode='min': We're monitoring a metric we want to minimize (loss)
# patience=3: Wait 3 epochs without improvement before reducing LR
# factor=0.5: Multiply learning rate by 0.5 when reducing
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',      # Minimize the monitored metric (loss)
    patience=3,      # Number of epochs to wait before reducing LR
    factor=0.5,      # Multiply LR by this factor when reducing
    verbose=True     # Print message when LR is reduced
)

print(f"Scheduler: {scheduler.__class__.__name__}")
print("\n" + "="*60)
print("Setup complete! Ready to train.")
print("="*60)

# 🚀 Training Loop

## The Training Process:
Each **epoch** is one complete pass through the entire training dataset.

### Within Each Epoch:
1. **Forward Pass**: Feed data through the network to get predictions
2. **Loss Calculation**: Measure how wrong the predictions are
3. **Backward Pass**: Calculate gradients (how to adjust weights)
4. **Weight Update**: Adjust weights to reduce loss
5. **Repeat**: Process all batches in the dataset

### Key Concepts:
- **Batch**: A subset of training data processed together
- **Gradient**: The direction and magnitude of weight adjustments
- **Backpropagation**: Algorithm for computing gradients efficiently

In [ ]:
# ============================================================================
# TRAINING LOOP
# ============================================================================

# Number of complete passes through the training dataset
num_epochs = 15

print("Starting training...")
print("="*60)

for epoch in range(num_epochs):
    # ========================================================================
    # TRAINING MODE
    # ========================================================================
    # Set model to training mode
    # This enables dropout and batch normalization training behavior
    model.train()
    
    # Initialize metrics for this epoch
    train_loss = 0.0        # Cumulative loss
    correct = 0             # Number of correct predictions
    total = 0               # Total number of samples processed
    
    # ========================================================================
    # BATCH PROCESSING
    # ========================================================================
    # Iterate through all batches in the training set
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data and labels to device (GPU/CPU)
        # data: images, shape (batch_size, 1, 28, 28)
        # target: labels, shape (batch_size,)
        data, target = data.to(device), target.to(device)
        
        # --------------------------------------------------------------------
        # STEP 1: ZERO GRADIENTS
        # --------------------------------------------------------------------
        # Clear gradients from previous iteration
        # PyTorch accumulates gradients by default, so we must clear them
        optimizer.zero_grad()
        
        # --------------------------------------------------------------------
        # STEP 2: FORWARD PASS
        # --------------------------------------------------------------------
        # Pass batch through the network to get predictions
        # output shape: (batch_size, 10) - raw scores for each class
        output = model(data)
        
        # Calculate loss (how wrong are our predictions?)
        # Compares predicted scores with true labels
        loss = criterion(output, target)
        
        # --------------------------------------------------------------------
        # STEP 3: BACKWARD PASS
        # --------------------------------------------------------------------
        # Compute gradients using backpropagation
        # This calculates how each weight contributed to the loss
        loss.backward()
        
        # --------------------------------------------------------------------
        # STEP 4: WEIGHT UPDATE
        # --------------------------------------------------------------------
        # Update model weights using computed gradients
        # The optimizer applies the learning algorithm (Adam in our case)
        optimizer.step()
        
        # --------------------------------------------------------------------
        # TRACK METRICS
        # --------------------------------------------------------------------
        # Accumulate loss for epoch statistics
        train_loss += loss.item()
        
        # Calculate accuracy
        # torch.max returns (values, indices) - we want indices (predicted class)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)                          # Add batch size
        correct += (predicted == target).sum().item()    # Count correct predictions
        
        # Print progress every 100 batches
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, '
                  f'Batch {batch_idx}/{len(train_loader)}, '
                  f'Loss: {loss.item():.4f}')
    
    # ========================================================================
    # EPOCH STATISTICS
    # ========================================================================
    # Calculate average loss and accuracy for the entire epoch
    avg_loss = train_loss / len(train_loader)      # Average loss per batch
    accuracy = 100 * correct / total                # Accuracy percentage
    
    print(f'\n{"="*60}')
    print(f'Epoch {epoch+1} Complete')
    print(f'Average Loss: {avg_loss:.4f}')
    print(f'Training Accuracy: {accuracy:.2f}%')
    print(f'{"="*60}\n')
    
    # ========================================================================
    # LEARNING RATE SCHEDULING
    # ========================================================================
    # Adjust learning rate based on training progress
    # If loss hasn't improved for 'patience' epochs, reduce LR
    scheduler.step(avg_loss)

print("\n" + "="*60)
print("Training complete!")
print("="*60)

# 🎯 Make Predictions and Create Submission

## Inference (Prediction) Process:
1. **Evaluation Mode**: Disable dropout and batch normalization training behavior
2. **No Gradient Computation**: Save memory and speed up inference
3. **Batch Processing**: Process test data in batches
4. **Extract Predictions**: Get the class with highest probability
5. **Create Submission**: Format predictions for Kaggle submission

In [ ]:
# ============================================================================
# INFERENCE (MAKING PREDICTIONS)
# ============================================================================

# Set model to evaluation mode
# This disables dropout and changes batch normalization behavior
# Important: Always use eval() mode for inference!
model.eval()

# Store all predictions
predictions = []

print("Making predictions on test data...")
print("="*60)

# Disable gradient computation for inference
# Benefits: Faster computation, lower memory usage
# We don't need gradients because we're not training
with torch.no_grad():
    for batch_idx, data in enumerate(test_loader):
        # Move batch to device (GPU/CPU)
        data = data.to(device)
        
        # Forward pass: get predictions
        # output shape: (batch_size, 10) - raw scores for each class
        output = model(data)
        
        # Extract predicted class
        # torch.max returns (max_values, max_indices)
        # We want indices (which class has highest score)
        _, predicted = torch.max(output.data, 1)
        
        # Move predictions to CPU and convert to numpy
        # Extend list with all predictions in this batch
        predictions.extend(predicted.cpu().numpy())
        
        # Print progress every 50 batches
        if batch_idx % 50 == 0:
            print(f"Processed {batch_idx}/{len(test_loader)} batches")

print(f"Total predictions made: {len(predictions)}")

# ============================================================================
# CREATE SUBMISSION FILE
# ============================================================================

# Format predictions according to Kaggle requirements
# ImageId: Sequential ID starting from 1
# Label: Predicted digit (0-9)
submission = pd.DataFrame({
    'ImageId': range(1, len(predictions) + 1),
    'Label': predictions
})

# Save to CSV file
submission.to_csv('submission.csv', index=False)

print("="*60)
print("Submission file created successfully!")
print(f"File: submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nFirst few predictions:")
print(submission.head(10))
print("="*60)

# ============================================================================
# QUICK STATISTICS
# ============================================================================
print("\nPrediction Distribution:")
print(submission['Label'].value_counts().sort_index())
print("\n" + "="*60)
print("Ready to submit to Kaggle!")
print("="*60)

# 📊 Optional: Visualize Predictions

Let's visualize some predictions to understand model behavior

In [ ]:
# ============================================================================
# VISUALIZE SAMPLE PREDICTIONS
# ============================================================================

# Display a grid of test images with predictions
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle('Sample Predictions from Test Set', fontsize=16, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    # Get image and prediction
    img = X_test[idx].reshape(28, 28)
    predicted_digit = predictions[idx]
    
    # Display image
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Predicted: {predicted_digit}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Visualization complete!")

# 💾 Optional: Save and Load Model

Learn how to save your trained model and load it later

In [ ]:
# ============================================================================
# SAVE TRAINED MODEL
# ============================================================================

# Save model weights (recommended method)
# This saves only the learned parameters, not the architecture
torch.save(model.state_dict(), 'digit_cnn_weights.pth')
print("✓ Model weights saved to: digit_cnn_weights.pth")

# Save complete model (architecture + weights)
# Useful but creates larger files
torch.save(model, 'digit_cnn_complete.pth')
print("✓ Complete model saved to: digit_cnn_complete.pth")

print("\n" + "="*60)

# ============================================================================
# LOAD TRAINED MODEL
# ============================================================================

# Method 1: Load weights into existing model (recommended)
# First create a new model instance
loaded_model = DigitCNN().to(device)
# Then load the saved weights
loaded_model.load_state_dict(torch.load('digit_cnn_weights.pth'))
loaded_model.eval()  # Set to evaluation mode
print("✓ Model weights loaded successfully")

# Method 2: Load complete model
# loaded_model = torch.load('digit_cnn_complete.pth')
# loaded_model.eval()

print("="*60)
print("\nModel ready for inference!")

# 📚 Deep Learning Template - Key Takeaways

## 🎓 What You've Learned:

### 1. **Data Pipeline**
- Loading and preprocessing image data
- Normalizing inputs (critical for neural network training)
- Creating custom PyTorch Datasets
- Using DataLoaders for efficient batch processing

### 2. **CNN Architecture**
- Convolutional layers for feature extraction
- Pooling layers for dimensionality reduction
- Dropout for regularization (preventing overfitting)
- Fully connected layers for classification

### 3. **Training Process**
- Forward pass: predictions
- Loss calculation: measuring error
- Backward pass: computing gradients
- Weight updates: improving the model

### 4. **Best Practices**
- Use GPU when available (`device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`)
- Normalize input data (divide by 255 for images)
- Use dropout to prevent overfitting
- Monitor training metrics (loss, accuracy)
- Use learning rate schedulers for better convergence
- Set model to eval() mode for inference

---

## 🔧 Common Modifications for Your Projects:

### Change Input Size:
```python
# For different image sizes (e.g., 64x64 RGB images):
X_train = X_train.reshape(-1, 3, 64, 64).astype('float32') / 255.0
# Update first conv layer:
self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 3 channels for RGB
```

### Change Number of Classes:
```python
# For different number of output classes:
self.fc2 = nn.Linear(128, num_classes)  # Replace 10 with your number
```

### Add Data Augmentation:
```python
transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
])
train_dataset = MNISTDataset(X_train, y_train, transform=transform)
```

### Adjust Hyperparameters:
```python
batch_size = 128           # Larger batch = faster training, more memory
learning_rate = 0.0001     # Smaller LR = more stable, slower training
num_epochs = 30            # More epochs = more training time
```

---

## 🚀 Next Steps:

1. **Experiment with architecture**: Add more layers, try different filter sizes
2. **Tune hyperparameters**: Learning rate, batch size, dropout rates
3. **Add validation set**: Monitor performance on unseen data during training
4. **Implement early stopping**: Stop training when validation performance plateaus
5. **Try different optimizers**: SGD with momentum, AdamW, RMSprop
6. **Add data augmentation**: Improve generalization with transformed images
7. **Use pre-trained models**: Transfer learning with ResNet, VGG, etc.

---

## 📖 PyTorch Documentation:
- **Official Docs**: https://pytorch.org/docs/
- **Tutorials**: https://pytorch.org/tutorials/
- **Forums**: https://discuss.pytorch.org/

---

## 🎯 Template Usage:
Use this notebook as a starting point for any image classification project. Simply adjust:
1. Data loading (replace CSV with your data source)
2. Image dimensions and channels
3. Number of output classes
4. Model architecture (if needed)
5. Hyperparameters

**Good luck with your deep learning journey! 🚀**